# Spacedust: *de novo* discovery of conserved gene clusters in microbial genomes
---

<img src="https://raw.githubusercontent.com/soedinglab/spacedust/master/.github/spacedust.png" height="200" align="right" style="height:200px"/>

Spacedust 是一个用于在多基因组之间识别保守基因簇的模块化工具包，结合了 Foldseek 与 MMseqs2 的快速结构/同源搜索能力。本笔记本版本已针对本地环境整理，默认从 `example/` 目录读取输入文件，可在下方“配置运行参数”单元中调整路径与选项。

**Input**

* 每个文件应包含一个基因组，可使用 `.fna`（自动运行 prodigal 预测蛋白）或 `.faa`（已含 Prodigal header 的蛋白序列）格式。
* 默认会将 `example/` 目录下匹配的文件复制到运行所需的 `jobname_input/`（以及在需要时的 `jobname_target/`）目录。

**Output**

* Spacedust 运行后会生成以 `jobname` 为前缀的 TSV 结果文件及辅助数据，并可在后续可视化步骤中复用。

## 工作流概览

1. 配置运行参数与数据路径。
2. 准备输入目录（从本地 `example/` 复制到 `jobname_*`）。
3. 下载或恢复 Spacedust 依赖及数据库。
4. 执行 Spacedust 主流程。
5. 浏览结果并进行可视化。

> 提示：按顺序运行各个单元。如需重新开始，可清理生成的 `jobname_*`、`database/` 与 `tmp/` 目录后重跑。

In [8]:
#@title 配置运行参数
from pathlib import Path

jobname = "spacedust_example" #@param {type:"string"}
input_mode = "all-against-all" #@param ["query-target", "all-against-all"]
target_db = "self-uploaded" #@param ["self-uploaded", "KEGG_70"]
search_mode = "MMseqs2" #@param ["MMseqs2", "Foldseek"]
run_prodigal = True #@param {type:"boolean"}
max_gene_gap = 3 #@param {type:"integer"}
num_iterations = 1 #@param {type:"integer"}

#@markdown ---
#@markdown 本地输入设置
base_input_dir = "example" #@param {type:"string"}
target_dir = "" #@param {type:"string"}
selected_query_genome_name = "" #@param {type:"string"}

#@markdown ---
#@markdown 本地依赖设置（若已手动下载）
use_local_dependencies = True #@param {type:"boolean"}
spacedust_binary_path = "/mnt/f/OneDrive/文档（科研）/脚本/Download/13-A.baumannii/5-Spacedust/download/spacedust/bin/spacedust" #@param {type:"string"}
prodigal_binary_path = "/mnt/f/OneDrive/文档（科研）/脚本/Download/13-A.baumannii/5-Spacedust/download/prodigal/bin/prodigal.linux" #@param {type:"string"}
foldseek_binary_path = "/mnt/f/OneDrive/文档（科研）/脚本/Download/13-A.baumannii/5-Spacedust/download/foldseek/bin/foldseek" #@param {type:"string"}
external_db_root = "/mnt/e/Scientifc_software/KEGG_70" #@param {type:"string"}

jobname = "".join(jobname.split())
if not jobname:
    raise ValueError("jobname 不能为空。")

base_input_dir = Path(base_input_dir).expanduser()
if not base_input_dir.exists():
    raise FileNotFoundError(f"未找到输入目录: {base_input_dir.resolve()}")

target_dir = Path(target_dir).expanduser() if target_dir else None
if target_dir is not None and not target_dir.exists():
    raise FileNotFoundError(f"未找到目标目录: {target_dir.resolve()}")

selected_query_genome_name = selected_query_genome_name.strip() or None

input_type = 1 if input_mode == "all-against-all" else 0
target_type = 0 if target_db == "self-uploaded" else 1
search_type = 0 if search_mode == "MMseqs2" else 1

use_local_dependencies = bool(use_local_dependencies)

spacedust_binary_path = Path(spacedust_binary_path).expanduser() if spacedust_binary_path.strip() else None
prodigal_binary_path = Path(prodigal_binary_path).expanduser() if prodigal_binary_path.strip() else None
foldseek_binary_path = Path(foldseek_binary_path).expanduser() if foldseek_binary_path.strip() else None
external_db_root = Path(external_db_root).expanduser() if external_db_root.strip() else None

if use_local_dependencies:
    if not spacedust_binary_path or not spacedust_binary_path.is_file():
        raise FileNotFoundError("use_local_dependencies=True，但未找到 spacedust 可执行文件。")
    if run_prodigal and (not prodigal_binary_path or not prodigal_binary_path.is_file()):
        raise FileNotFoundError("use_local_dependencies=True，但未找到 prodigal 可执行文件。")
    if search_type == 1 and (not foldseek_binary_path or not foldseek_binary_path.is_file()):
        raise FileNotFoundError("search_mode=Foldseek 时需要提供 foldseek 可执行文件路径。")
    if target_type == 1 and (not external_db_root or not external_db_root.exists()):
        raise FileNotFoundError("需要提供 KEGG 数据库目录 external_db_root。")

spacedust_binary_path_str = str(spacedust_binary_path) if spacedust_binary_path else ""
prodigal_binary_path_str = str(prodigal_binary_path) if prodigal_binary_path else ""
foldseek_binary_path_str = str(foldseek_binary_path) if foldseek_binary_path else ""
external_db_root_str = str(external_db_root) if external_db_root else ""

print(f"Job name: {jobname}")
print(f"查询文件目录: {base_input_dir.resolve()}")
if input_type == 0 and target_type == 0:
    if target_dir is None:
        raise ValueError("query-target 模式需要指定 target_dir。")
    print(f"目标文件目录: {target_dir.resolve()}")
elif target_type == 1:
    print(f"使用预置数据库: {target_db}")

if use_local_dependencies:
    print("使用本地依赖：")
    print(f"  spacedust: {spacedust_binary_path_str}")
    if run_prodigal:
        print(f"  prodigal: {prodigal_binary_path_str}")
    if search_type == 1:
        print(f"  foldseek: {foldseek_binary_path_str}")
    if target_type == 1:
        print(f"  预置数据库目录: {external_db_root_str}")
else:
    print("将尝试在线下载所需依赖。")


Job name: spacedust_example
查询文件目录: /mnt/c/Users/Administrator/Desktop/example
使用本地依赖：
  spacedust: /mnt/f/OneDrive/文档（科研）/脚本/Download/13-A.baumannii/5-Spacedust/download/spacedust/bin/spacedust
  prodigal: /mnt/f/OneDrive/文档（科研）/脚本/Download/13-A.baumannii/5-Spacedust/download/prodigal/bin/prodigal.linux


In [1]:
#@title 准备输入目录
import shutil

job_input_dir = Path(f"{jobname}_input")
job_target_dir = Path(f"{jobname}_target")

for path in (job_input_dir, job_target_dir):
    if path.exists():
        shutil.rmtree(path)

job_input_dir.mkdir(parents=True, exist_ok=True)

query_patterns = ("*.fna", "*.faa", "*.fasta", "*.fa")
query_files = []
for pattern in query_patterns:
    query_files.extend(sorted(base_input_dir.glob(pattern)))

if not query_files:
    raise FileNotFoundError(f"在 {base_input_dir.resolve()} 中未找到 `.fna` 或 `.faa` 文件。")

for src in query_files:
    shutil.copy2(src, job_input_dir / src.name)

print(f"已复制 {len(query_files)} 个查询文件到 {job_input_dir.resolve()}")

if input_type == 0:
    if target_type == 0:
        if target_dir is None:
            raise ValueError("query-target 模式需要目标目录。")
        job_target_dir.mkdir(parents=True, exist_ok=True)
        target_files = []
        for pattern in query_patterns:
            target_files.extend(sorted(target_dir.glob(pattern)))
        if not target_files:
            raise FileNotFoundError(f"在 {target_dir.resolve()} 中未找到 `.fna` 或 `.faa` 文件。")
        for src in target_files:
            shutil.copy2(src, job_target_dir / src.name)
        print(f"已复制 {len(target_files)} 个目标文件到 {job_target_dir.resolve()}")
    else:
        print(f"将使用预置数据库 {target_db} 作为目标。")
else:
    print("全对全比对模式，仅使用查询集合。")

NameError: name 'Path' is not defined

In [7]:
#@title 下载依赖与数据库
%%bash -s "$jobname" "$input_type" "$search_type" "$run_prodigal" "$target_type" "$target_db" "$use_local_dependencies" "$spacedust_binary_path_str" "$prodigal_binary_path_str" "$foldseek_binary_path_str" "$external_db_root_str"
JOBNAME=$1
INPUT_TYPE=$2
SEARCH_TYPE=$3
NEED_PRODIGAL=$4
TARGET_TYPE=$5
TARGET_DB=$6
USE_LOCAL=$7
SPACEDUST_BIN=$8
PRODIGAL_BIN=$9
FOLDSEEK_BIN=${10}
EXTERNAL_DB_ROOT=${11}

set -euo pipefail

mkdir -p spacedust/bin

if [ "$USE_LOCAL" = "True" ]; then
  if [ ! -f "$SPACEDUST_BIN" ]; then
    echo "未找到本地 spacedust 可执行文件: $SPACEDUST_BIN" >&2
    exit 1
  fi
  ln -sf "$SPACEDUST_BIN" spacedust/bin/spacedust
  chmod +x spacedust/bin/spacedust
  touch SPACEDUST_READY
elif [ ! -e SPACEDUST_READY ]; then
  wget -q https://mmseqs.com/spacedust/spacedust-linux-avx2.tar.gz
  tar -xzf spacedust-linux-avx2.tar.gz
  rm -f spacedust-linux-avx2.tar.gz
  touch SPACEDUST_READY
fi

if [ "$SEARCH_TYPE" = "1" ]; then
  if [ "$USE_LOCAL" = "True" ]; then
    if [ ! -f "$FOLDSEEK_BIN" ]; then
      echo "未找到本地 foldseek 可执行文件: $FOLDSEEK_BIN" >&2
      exit 1
    fi
    ln -sf "$FOLDSEEK_BIN" spacedust/bin/foldseek
    chmod +x spacedust/bin/foldseek
    touch FOLDSEEK_READY
  elif [ ! -e FOLDSEEK_READY ]; then
    wget -q https://mmseqs.com/foldseek/foldseek-linux-avx2.tar.gz
    tar -xzf foldseek-linux-avx2.tar.gz
    rm -f foldseek-linux-avx2.tar.gz
    mv foldseek/bin/foldseek spacedust/bin/
    rm -rf foldseek
    touch FOLDSEEK_READY
  fi
fi

if [ "$NEED_PRODIGAL" = "True" ]; then
  if [ "$USE_LOCAL" = "True" ]; then
    if [ ! -f "$PRODIGAL_BIN" ]; then
      echo "未找到本地 prodigal 可执行文件: $PRODIGAL_BIN" >&2
      exit 1
    fi
    ln -sf "$PRODIGAL_BIN" spacedust/bin/prodigal
    chmod +x spacedust/bin/prodigal
    touch PRODIGAL_READY
  elif [ ! -e PRODIGAL_READY ]; then
    wget -q https://github.com/hyattpd/Prodigal/releases/download/v2.6.3/prodigal.linux -O spacedust/bin/prodigal
    chmod +x spacedust/bin/prodigal
    touch PRODIGAL_READY
  fi
fi

mkdir -p database
cd database
if [ "$INPUT_TYPE" = "0" ] && [ "$TARGET_TYPE" = "1" ]; then
  if [ "$USE_LOCAL" = "True" ] && [ -n "$EXTERNAL_DB_ROOT" ]; then
    if [ ! -d "$EXTERNAL_DB_ROOT" ]; then
      echo "未找到 KEGG 数据库目录: $EXTERNAL_DB_ROOT" >&2
      exit 1
    fi
    for path in "$EXTERNAL_DB_ROOT"/keggclusterdb*; do
      [ -e "$path" ] || continue
      ln -sf "$path" "$(basename "$path")"
    done
    touch ../SPACEDUST_DB_READY
  elif [ ! -e ../SPACEDUST_DB_READY ]; then
    wget -N -np -nv http://wwwuser.gwdg.de/~compbiol/spacedust/${TARGET_DB}.tar.gz
    tar -xzf ${TARGET_DB}.tar.gz
    rm -f ${TARGET_DB}.tar.gz
    touch ../SPACEDUST_DB_READY
  fi
fi


SyntaxError: invalid syntax (47575250.py, line 3)

In [ ]:
#@title 运行 Spacedust
%%bash -s $jobname $input_type $search_type $run_prodigal $target_type $max_gene_gap $num_iterations
JOBNAME=$1
INPUT_TYPE=$2
SEARCH_TYPE=$3
NEED_PRODIGAL=$4
TARGET_TYPE=$5
MAX_GENE_GAP=$6
NUM_ITERATIONS=$7

set -euo pipefail

mkdir -p database tmp

if [ "${NEED_PRODIGAL}" = "True" ]; then
  shopt -s nullglob
  for filename in ${JOBNAME}_input/*.fna; do
    base=$(basename "$filename" .fna)
    if [ ! -f "${JOBNAME}_input/${base}.faa" ]; then
      spacedust/bin/prodigal -i "$filename" -a "${JOBNAME}_input/${base}.faa" >/dev/null 2>&1
    fi
  done
  if [ "${INPUT_TYPE}" = "0" ] && [ "${TARGET_TYPE}" = "0" ]; then
    for filename in ${JOBNAME}_target/*.fna; do
      base=$(basename "$filename" .fna)
      if [ ! -f "${JOBNAME}_target/${base}.faa" ]; then
        spacedust/bin/prodigal -i "$filename" -a "${JOBNAME}_target/${base}.faa" >/dev/null 2>&1
      fi
    done
  fi
  shopt -u nullglob
fi

shopt -s nullglob
spacedust/bin/spacedust createsetdb ${JOBNAME}_input/*.faa database/${JOBNAME}_input tmp -v 0 >/dev/null

if [ "${INPUT_TYPE}" = "1" ]; then
  spacedust/bin/spacedust clustersearch database/${JOBNAME}_input database/${JOBNAME}_input "${JOBNAME}" tmp --filter-self-match --search-mode "${SEARCH_TYPE}" --max-gene-gap "${MAX_GENE_GAP}" -v 0 >/dev/null
else
  if [ "${TARGET_TYPE}" = "0" ]; then
    spacedust/bin/spacedust createsetdb ${JOBNAME}_target/*.faa database/${JOBNAME}_db tmp -v 0 >/dev/null
    spacedust/bin/spacedust clustersearch database/${JOBNAME}_input database/${JOBNAME}_db "${JOBNAME}" tmp --search-mode "${SEARCH_TYPE}" --max-gene-gap "${MAX_GENE_GAP}" -v 0 >/dev/null
  else
    spacedust/bin/spacedust clustersearch database/${JOBNAME}_input database/keggclusterdb "${JOBNAME}" tmp --search-mode "${SEARCH_TYPE}" --max-gene-gap "${MAX_GENE_GAP}" -v 0 >/dev/null
  fi
fi
shopt -u nullglob

spacedust/bin/spacedust prefixid tmp/latest/clusters "${JOBNAME}_pref" --tsv -v 0 >/dev/null

awk '{ print $2"\t"$3 }' "${JOBNAME}_pref" > qid_tid
awk '{ print $1 }' "${JOBNAME}_pref" > cluid
rm "${JOBNAME}_pref"

awk 'BEGIN{OFS=FS="\t"} NR==FNR{clr[$1]=$2; next} {$1=clr[$1]; print}' database/${JOBNAME}_input.lookup qid_tid > tmp_qname
if [ "${INPUT_TYPE}" = "1" ]; then
  awk 'BEGIN{OFS=FS="\t"} NR==FNR{clr[$1]=$2; next} {$2=clr[$2]; print}' database/${JOBNAME}_input.lookup tmp_qname > qname_tname
elif [ "${TARGET_TYPE}" = "0" ]; then
  awk 'BEGIN{OFS=FS="\t"} NR==FNR{clr[$1]=$2; next} {$2=clr[$2]; print}' database/${JOBNAME}_db.lookup tmp_qname > qname_tname
else
  awk 'BEGIN{OFS=FS="\t"} NR==FNR{clr[$1]=$2; next} {$2=clr[$2]; print}' database/keggclusterdb.lookup tmp_qname > qname_tname
fi

sed -i 's/NZ_/NZ./g' qname_tname
sed -i 's/NC_/NC./g' qname_tname
tr '_' '\t' < qname_tname > qname_tname_sep
rm tmp_qname qname_tname

awk 'BEGIN{OFS=FS="\t"} NR==FNR{clr[$1]=$3; next} {$1=clr[$1]; print}' database/${JOBNAME}_input.lookup qid_tid > tmp_qset
if [ "${INPUT_TYPE}" = "1" ]; then
  awk 'BEGIN{OFS=FS="\t"} NR==FNR{clr[$1]=$3; next} {$2=clr[$2]; print}' database/${JOBNAME}_input.lookup tmp_qset > qset_tset
elif [ "${TARGET_TYPE}" = "0" ]; then
  awk 'BEGIN{OFS=FS="\t"} NR==FNR{clr[$1]=$3; next} {$2=clr[$2]; print}' database/${JOBNAME}_db.lookup tmp_qset > qset_tset
else
  awk 'BEGIN{OFS=FS="\t"} NR==FNR{clr[$1]=$3; next} {$2=clr[$2]; print}' database/keggclusterdb.lookup tmp_qset > qset_tset
fi
rm tmp_qset

paste cluid qset_tset qid_tid qname_tname_sep > "${JOBNAME}_plot"
rm qid_tid qname_tname_sep qset_tset cluid

spacedust/bin/spacedust prefixid database/${JOBNAME}_input database/${JOBNAME}_input_pref --tsv -v 0 >/dev/null

SyntaxError: invalid syntax (2568337159.py, line 3)

In [ ]:
#@title 浏览 cluster matches 摘要
from pathlib import Path
from IPython.display import HTML, display

result_path = Path(jobname)
if not result_path.exists():
    print(f"未找到结果文件: {result_path.resolve()}")
else:
    lines = [line for line in result_path.read_text().splitlines() if line]
    if not lines:
        print(f"结果文件 {result_path} 为空。")
    else:
        html = [
            "<style>tbody tr.head {font-weight:bold;background-color:#f2f2f2;}</style>",
            "<table style='text-align:left'>",
            "  <thead>",
            "    <tr>",
            "      <th>Cluster Match ID</th>",
            "      <th>Query Acc.</th>",
            "      <th>Target Acc.</th>",
            "      <th>Cluster Match Pval</th>",
            "      <th colspan=\"8\">Num. Hits</th>",
            "    </tr>",
            "    <tr>",
            "      <th colspan='2'>Query ID</th>",
            "      <th>Target ID</th>",
            "      <th>SeqId</th>",
            "      <th>Eval</th>",
            "      <th>qStart</th>",
            "      <th>qEnd</th>",
            "      <th>qLength</th>",
            "      <th>tStart</th>",
            "      <th>tEnd</th>",
            "      <th>tLength</th>",
            "      <th>Aln Cigar</th>",
            "    </tr>",
            "  </thead>",
            "  <tbody>",
        ]
        for line in lines:
            if line.startswith('#'):
                cols = line.split('	')
                if len(cols) == 6:
                    html.append("<tr class='head'>")
                    html.append(f"<td>{cols[0][1:]}</td>")
                    html.append(f"<td>{cols[1]}</td>")
                    html.append(f"<td>{cols[2]}</td>")
                    html.append(f"<td>{cols[4]}</td>")
                    html.append(f"<td colspan='8'>{cols[5]}</td>")
                    html.append("</tr>")
            elif line.startswith('>'):
                cols = line.split('	')
                if len(cols) == 12:
                    html.append("<tr>")
                    html.append(f"<td colspan='2'>{cols[0][1:]}</td>")
                    html.append(f"<td>{cols[1]}</td>")
                    html.append(f"<td>{cols[3]}</td>")
                    html.append(f"<td>{cols[4]}</td>")
                    html.append(f"<td>{cols[5]}</td>")
                    html.append(f"<td>{cols[6]}</td>")
                    html.append(f"<td>{cols[7]}</td>")
                    html.append(f"<td>{cols[8]}</td>")
                    html.append(f"<td>{cols[9]}</td>")
                    html.append(f"<td>{cols[10]}</td>")
                    html.append(f"<td>{cols[11]}</td>")
                    html.append("</tr>")
        html.append("  </tbody>")
        html.append("</table>")
        display(HTML('
'.join(html)))

In [ ]:
#@title 列出输出文件
from pathlib import Path

result_prefix = Path(jobname).name
paths = sorted(Path('.').glob(f"{result_prefix}*"))
if not paths:
    print("暂未找到以 jobname 开头的输出文件。")
else:
    for path in paths:
        if path.is_file():
            size_kb = path.stat().st_size / 1024
            print(f"{path} - {size_kb:.1f} KiB")
        else:
            print(f"{path}/")

# **Visualization**


In [ ]:
#@title 准备可视化所需数据
import importlib
import numpy as np
import pandas as pd
import seaborn as sns

missing = []
for package, import_name in [("ipympl", "ipympl"),("rpy2", "rpy2")]:
    try:
        importlib.import_module(import_name)
    except ImportError:
        missing.append(package)

if missing:
    raise ImportError(f"缺少以下 Python 包，请先在本地环境中安装：{', '.join(missing)}")

matchhit = pd.read_csv(f"{jobname}_plot", sep="	", names=['cluid','qsetid', 'tsetid', 'qseqid','tseqid','qname', 'qid_p','qid','qstart','qend', 'tname','tid','tstart','tend'])
lookup = pd.read_csv(f"database/{jobname}_input.lookup", sep="	", names=['seqid','header','setid'])
all_seq = pd.read_csv(f"database/{jobname}_input_pref", sep="	", names=['id','seq'], header=None, dtype={'id' : int, 'seq': str})


In [ ]:
#@title 选择查询基因组并构建索引
import numpy as np

df = pd.read_csv(f'database/{jobname}_input.source', sep="	", names=['query_genome_id','query_genome_name'], dtype={'query_genome_id': int, 'query_genome_name': str})
available = df['query_genome_name'].tolist()
if not available:
    raise ValueError('未在 source 文件中找到任何基因组。')

if selected_query_genome_name is None:
    chosen = available[0]
elif selected_query_genome_name not in available:
    raise ValueError(f"selected_query_genome_name={selected_query_genome_name} 不在可选列表中: {available}")
else:
    chosen = selected_query_genome_name

selected_query_genome_name = chosen
query_genome = int(df.loc[df['query_genome_name'] == chosen, 'query_genome_id'].iloc[0])
print(f"使用查询基因组: {chosen} (ID={query_genome})")

matchhit_temp = matchhit[matchhit['qsetid'] == query_genome]
qid = matchhit_temp.drop_duplicates(['qid','tsetid'], keep='last')['qid'].to_numpy()
matchhit_array = np.zeros(qid.max()+1, dtype=int) if len(qid) else np.array([])
for i in qid:
    matchhit_array[i] += 1

ordered = matchhit.sort_values(['cluid', 'qid'])
qid_all = ordered['qid'].to_numpy()
tid_all = ordered['tid'].to_numpy()
cluid_all = ordered['cluid'].to_numpy()
matchpair_array = np.zeros(qid_all.max(), dtype=int) if len(qid_all) else np.array([])
for i in np.arange(len(qid_all)-1):
    if cluid_all[i] == cluid_all[i+1]:
        if qid_all[i] == qid_all[i+1] - 1:
            matchpair_array[qid_all[i]] += 1
        else:
            if abs(qid_all[i+1] - qid_all[i]) == abs(tid_all[i+1] - tid_all[i]):
                for x in np.arange(qid_all[i], qid_all[i+1]):
                    matchpair_array[x] += 1

count = np.zeros(matchhit_array.max()+1) if matchhit_array.size else np.array([])
for value in matchhit_array.tolist():
    count[value] += 1

lookup_temp = lookup[lookup['setid'] == query_genome].copy()
lookup_temp['idx'] = lookup_temp['header'].str.split('_').str[-3].astype(int)
lookup_temp['qstart'] = lookup_temp['header'].str.split('_').str[-2].astype(int)
lookup_temp['qend'] = lookup_temp['header'].str.split('_').str[-1].astype(int)

In [ ]:
#@title Cluster matches heatmap/bar plot

Zoom = False #@param {type:"boolean"}
#@markdown Zoom in to the plot by setting a lower and upper bound of the protein id (ignored if `zoom` is not selected)
lower_bound = 1 #@param {type:"integer"}
upper_bound = 100 #@param {type:"integer"}

# Ensure that lower_bound is always smaller than upper_bound
if lower_bound >= upper_bound:
    raise ValueError("Lower bound must be smaller than upper bound")


#combined bar plot
%matplotlib widget

import matplotlib.pyplot as plt
import math
import matplotlib.cm as cm
from matplotlib.widgets import Slider
import matplotlib.ticker as ticker

# from matplotlib.colors import ListedColormap
from ipywidgets import *

from mpl_toolkits.axes_grid1 import make_axes_locatable

# Get the maximum protein ID and genome ID
max_protein_id = np.max(matchhit['qid'])
max_genome_id = np.max(matchhit['tsetid'])

# Create an empty matrix of the correct size
matrix = np.zeros((max_genome_id + 1, max_protein_id + 1))
matrix[query_genome,:] = 1
# Fill the matrix with the protein hits data
for _, row in matchhit.iterrows():
    matrix[row['tsetid'], row['qid']] = 1

# Create an empty matrix for the strand plot
strand_plot = np.zeros( len(lookup_temp['idx']), dtype=int)

# Loop through the gene dataframe to mark positive and negative strands
for _, row in lookup_temp.iterrows():
    if (row['qstart'] < row['qend']):
        # Set positive strand positions to 1
        strand_plot[row['idx']] = 1

# Create the figure and axes with adjusted height ratio
fig, (ax1, ax3, ax2) = plt.subplots(nrows=3, sharex=True, figsize=(10, 12), gridspec_kw={'height_ratios': [4, 0.1, 1]})

# Create the heatmap
heatmap = ax1.imshow(matrix, cmap='Blues', interpolation='none', aspect='auto')

# Set the title, y-axis label, and y-axis tick labels
# ax1.set_title('Presence/Absence Heatmap')
ax1.set_ylabel('Genome ID')
y_labels = range(0, max_genome_id + 1)
ax1.set_yticks(range(0, max_genome_id + 1))
ax1.set_yticklabels(y_labels)

# Use AutoLocator for x-axis ticks to adjust dynamically based on zoom level
ax1.xaxis.set_major_locator(ticker.AutoLocator())

# Use AutoLocator for y-axis ticks to adjust dynamically based on zoom level
ax1.yaxis.set_major_locator(ticker.AutoLocator())

# Set the x-axis tick labels and rotation
x_labels = range(0, max_protein_id + 1)
ax2.set_xticks(range(0, max_protein_id + 1))
ax2.set_xticklabels(x_labels, rotation=90)

# Add a bar plot below the heatmap
ax2.bar(np.arange(len(matchpair_array)), matchpair_array,width=1,align='edge',ecolor='black',color='lightpink')
ax2.bar(np.arange(len(matchhit_array)), matchhit_array,width=0.5,align='center')
ax2.set_ylabel('Hits Count')
ax2.set_xlabel('Query Protein Position Index')

# Adjust the limits of the x-axis to match the heatmap
if Zoom:
    ax2.set_xlim(lower_bound-0.5, upper_bound + 0.5)
else:
    ax2.set_xlim(-0.5, max_protein_id+1 - 0.5)

ax2.set_yscale('log')

# Use AutoLocator for x-axis ticks to adjust dynamically based on zoom level
ax2.xaxis.set_major_locator(ticker.AutoLocator())

# Add the strand plot as a thin line
cmap_binary = cm.binary
ax3.imshow(strand_plot.reshape(1, -1), cmap=cmap_binary, aspect='auto')
ax3.yaxis.set_ticks([])
ax3.set_ylabel('Strand')

# Enable interactive mode
plt.ion()

# Function to resize the plot when zooming in
def on_zoom(event):
    current_xlim = ax1.get_xlim()
    current_ylim = ax1.get_ylim()
    current_ylim2 = ax2.get_ylim()
    ax1.set_xlim(*current_xlim)
    ax1.set_ylim(*current_ylim)
    ax2.set_ylim(*current_ylim2)
    ax3.set_xlim(*current_xlim)

# Connect the on_zoom function to the zoom event
fig.canvas.mpl_connect('resize_event', on_zoom)

# Save the plot to a file (e.g., PNG)
plt.savefig('heatmap.pdf')

# # Display the saved plot in the notebook
# display(Image(filename='sample_plot.png'))
# Show the plot
plt.tight_layout()
plt.show()

In [ ]:
#@title Cluster matches bar plot
#combined bar plot
%matplotlib widget

import matplotlib.pyplot as plt
import math
from matplotlib.widgets import Slider
from ipywidgets import *

fig,ax=plt.subplots(figsize=(10,6))
fig.patch.set_facecolor('white')

# Simple mouse click function to store coordinates
def onclick(event):
    global ix, iy
    ix, iy = event.xdata, event.ydata

    # assign global variable to access outside of function
    global coords
    coords.append((ix, iy))

    # Disconnect after x clicks
    if len(coords) == 100:
        fig.canvas.mpl_disconnect(cid)
        plt.close(1)
    return

def on_move(event):
    if event.inaxes:
        print(f'data coords {event.xdata} {event.ydata},',
              f'pixel coords {event.x} {event.y}')


x = np.arange(qid.max()+1)
y1 = matchhit_array
y2 = matchpair_array

N=100

def bar(pos):
    pos = int(pos)
    ax.clear()
    if pos+N > len(x):
        n=len(x)-pos
    else:
        n=N
    X=x[pos:pos+n]
    Y=y1[pos:pos+n]
    Y2=y2[pos:pos+n]
    ax.bar(X,Y2,width=1,align='edge',ecolor='black',color='lightgrey')
    ax.bar(X,Y,width=0.5,align='edge',ecolor='black')

    # ax.axhline(10,color='r', linestyle='--')

#     ax.set_yscale('log')

barpos = plt.axes([0.18, 0.05, 0.55, 0.03], facecolor="skyblue")
slider = Slider(barpos, 'Gene pos', 50, len(x)-N, valinit=0)
slider.on_changed(bar)

bar(0)

coords = []

# Call click func
cid = fig.canvas.mpl_connect('button_press_event', onclick)

plt.show()

In [ ]:
#@title Select proteins(s) of interest, extract all cluster matches containing the protein-encoded gene(s).

#@markdown Input: Query Protein ID (single integer or range, e.g., '1' or '1-10')

# Query protein ID parameter
query_protein_id_input = '1' #@param {type:"string"}

# Function to parse the input string to get lower and upper bounds
def parse_input(input_str):
    parts = input_str.split('-')
    if len(parts) == 1:
        # Single integer
        return [int(parts[0])]
    elif len(parts) == 2:
        # Range
        return list(range(int(parts[0]), int(parts[1]) + 1))
    else:
        raise ValueError("Invalid input format. Please enter either a single integer or a range.")

# Print all numbers within the range
query_protein_id = parse_input(query_protein_id_input)

# Filter clusterid to include only clusters with all query_protein_ids
clusterid = matchhit.loc[matchhit['qid'].isin(query_protein_id), 'cluid'].unique().tolist()

# Check if all query_protein_ids are present in each cluster
filtered_clusterid = [cluster for cluster in clusterid if all(matchhit[matchhit['cluid'] == cluster]['qid'].isin(query_protein_id))]

appended_data = pd.DataFrame()
for i in clusterid:
    appended_data = pd.concat([appended_data, matchhit[matchhit['cluid'] == i]])

appended_data = appended_data[appended_data['qseqid'].map(appended_data['qseqid'].value_counts()) > 1]
# query_protein_id = 1 #@param {type:"integer"}
# clusterid = matchhit['cluid'][matchhit['qid'] ==query_protein_id].values.tolist()
# appended_data = pd.DataFrame()
# for i in clusterid:
#     appended_data = pd.concat([appended_data, matchhit[matchhit['cluid'] == i]])


In [ ]:
#@title Convert to R dataframe
%reload_ext rpy2.ipython
from rpy2.robjects import pandas2ri
# clusterid = matchhit['cluid'][matchhit['qid'] ==query_protein_id[0]].values.tolist()
# appended_data = pd.DataFrame()
# for i in clusterid:
#     appended_data = appended_data.append(matchhit[matchhit['cluid'] == i],ignore_index=True)

# condition = np.sign(appended_data['qstart']-appended_data['qend']) != np.sign(appended_data['tstart']-appended_data['tend'])
# appended_data['tstart'] = np.where(condition, -appended_data['tstart'],appended_data['tstart'])
# appended_data['tend'] = np.where(condition, -appended_data['tend'],appended_data['tend'])

predefined_qseqid = query_protein_id[0]

def invert_sign(group):
    matching_rows = group[group['qseqid'] == predefined_qseqid]

    if not matching_rows.empty:
        predefined_row = matching_rows.iloc[0]
        q_direction = np.sign(predefined_row['qstart'] - predefined_row['qend'])
        t_direction = np.sign(predefined_row['tstart'] - predefined_row['tend'])

        if q_direction != t_direction:
            group['tstart'], group['tend'] = -group['tstart'].values, -group['tend'].values

    return group

appended_data_g = appended_data.groupby('cluid', group_keys=False).apply(invert_sign)

center = str(query_protein_id[0])
appended_data_g = appended_data_g[appended_data_g['tname']!= appended_data_g['qname']]
gggene_df = appended_data_g.groupby(by="qid", as_index = False).first()[['qname','qid','qstart','qend']]
gggene_df = gggene_df.append(appended_data_g[['tname','qid','tstart','tend']].rename(columns={"tname": "qname", "tstart": "qstart","tend": "qend"})).reset_index()
pandas2ri.activate()
r_dataframe = pandas2ri.py2rpy(gggene_df)


In [ ]:
#@title 查看并保存基因组上下文图像
from pathlib import Path
from IPython.display import Image, display

image_path = Path("spacedust_plot.png")
if not image_path.exists():
    print(f"未找到图像文件: {image_path.resolve()}。请先运行上一个单元。")
else:
    print(f"图像已保存至: {image_path.resolve()}")
    display(Image(filename=str(image_path), width=500))

# 使用说明与提示
**执行顺序**
- 按“配置运行参数”→“准备输入目录”→“下载依赖与数据库”→“运行 Spacedust”的顺序依次执行。
- 若修改了输入文件或参数，请重新运行“准备输入目录”及其后的单元。

**本地依赖**
- 已手动下载的 spacedust、prodigal、foldseek 与 KEGG_70 目录在“配置运行参数”中填写后，会自动建立到当前工作目录的符号链接，不再重复下载。
- 如需改用其他路径，将布尔选项 `use_local_dependencies` 设为 False 或更新对应路径。

**故障排查**
- 若链接创建失败，请确认路径无误且对工作目录具有读取权限。
- 确认输入文件扩展名正确（`.fna`/`.faa`），并确保目录路径填写无误。
- 清理生成目录（如 `jobname_*`、`database/`、`tmp/`）后可重新开始一次全新的运行。

**反馈**
- 工具相关问题可提交至 https://github.com/soedinglab/spacedust/issues。


# **Instructions**
**Quick start**
1. Set parameters. Press "Runtime" -> "Run all".
2. Wait for file input box to appear below this cell to upload 1) all your query genome files and 2) your target genome files
3. The currently running step is indicated by a circle with a stop sign next to it.

**Result file contents**

1. Tab-separated text file (`.tsv`) of all reported cluster matches.
2. (If applicable) Protein sequences (`.faa`) predicted by prodigal.

<!--At the end of the job a download modal box will pop up with a `jobname.result.zip` file. Additionally, if the `save_to_google_drive` option was selected, the `jobname.result.zip` will be uploaded to your Google Drive. -->

**Troubleshooting**
<!--* If you wish to use ProtNLM for function annotation, check that the runtime type is set to GPU at "Runtime" -> "Change runtime type".-->
* Try to restart the session "Runtime" -> "Factory reset runtime".
* Check if your input genomes are unzipped.

**Limitations**
* Computing resources: Due to resource limitations, homology search is currently only available with MMseqs2 instead of Foldseek, which might affect sensitivity of detecting remote homology.

**Bugs**
- If you encounter any bugs, please report the issue to https://github.com/soedinglab/spacedust/issues